In [16]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\code\FSFM_Lite_Project


In [17]:
from src.models.fsfm_lite import FSFMLite
from src.models.threec_module import ThreeCModule, FusionBlock
from src.models.classifier_head import ClassifierHead
from src.datasets.celeba_spoof_dataset import (CelebASpoofDataset)
from torchvision import transforms

import pandas as pd
from pathlib import Path
import torch.nn as nn
import torch

In [18]:
model = FSFMLite()

print(model)

Using cache found in C:\Users\Admin/.cache\torch\hub\facebookresearch_dinov2_main


FSFMLite(
  (backbone): DINOBackbone(
    (backbone): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
        (norm): Identity()
      )
      (blocks): ModuleList(
        (0-11): 12 x NestedTensorBlock(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): MemEffAttention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (act): GELU(approximate='none')
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
            (drop): Dropout(p=0.0, inplace=Fal

In [19]:
PROJECT_ROOT = Path.cwd().parent

METADATA_ROOT = (
    PROJECT_ROOT
    / "metadata"
)

train_df = pd.read_csv(
    METADATA_ROOT
    / "train_df.csv"
)

test_df = pd.read_csv(
    METADATA_ROOT
    / "test_df.csv"
)

print(train_df.shape)
print(test_df.shape)

(244274, 4)
(25758, 4)


In [20]:
total_params = sum(
    p.numel() 
    for p in model.parameters()
)
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Params: {total_params:,}")
print(f"Trainable Params: {trainable_params:,}")

Total Params: 98,001,410
Trainable Params: 98,001,410


In [21]:
x = torch.randn(2, 3, 224, 224)
out = model(x)

print(out.shape)

torch.Size([2, 2])


In [22]:
face_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [23]:
dataset = CelebASpoofDataset(train_df, transform=face_transform)

sample = dataset[0]
image = sample["image"].unsqueeze(0)
with torch.no_grad():
    logits = model(image)

print(logits)
print(logits.shape)

tensor([[-0.0463,  0.1822]])
torch.Size([1, 2])


In [24]:
cls_token, patch_tokens = model.backbone(image)
enhanced_tokens = model.threec(patch_tokens, cls_token)
pooled = enhanced_tokens.mean(dim=1)

print(cls_token.shape)
print(patch_tokens.shape)

print(enhanced_tokens.shape)
print(pooled.shape)

torch.Size([1, 768])
torch.Size([1, 256, 768])
torch.Size([1, 256, 768])
torch.Size([1, 768])
